# Mini-Project 1 - Advanced Python Data Exploration & Automated Reporting

## Phase 0 - Setup
This notebook analyzes the Sample Superstore 2019 dataset using Pandas, NumPy, Matplotlib, and Seaborn.

In [ ]:
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path(r'C:\Users\o3995\Downloads\DEPI-R5\Technical\Projects\Mini-Project-1')
OUTPUT_DIR = PROJECT_DIR / 'output'
CHARTS_DIR = OUTPUT_DIR / 'charts'
RAW_DATA_PATH = PROJECT_DIR / 'Sample - Superstore 2019.csv'

for path in (OUTPUT_DIR, CHARTS_DIR):
    path.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')
logger = logging.getLogger('mini_project_1')

logger.info('Phase 0 setup complete')

## Phase 1 - Import & Inspect Data
Load the Excel dataset with exception handling, then inspect structure, metadata, missing values, and duplicate records.

In [ ]:
try:
    df = pd.read_csv(RAW_DATA_PATH, parse_dates=['Order Date', 'Ship Date'])
    logger.info('Loaded dataset with %s rows and %s columns', *df.shape)
except FileNotFoundError as exc:
    logger.error('Dataset file not found: %s', RAW_DATA_PATH)
    raise exc
except Exception as exc:
    logger.error('Could not load dataset: %s', exc)
    raise exc

df.head()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
df.tail()

### Initial Observations
- Review missing-value counts before choosing cleaning rules.
- Check date columns after loading to confirm they parsed correctly.
- Check duplicate count before removing records in Phase 3.
- Numeric summaries will guide outlier handling for Sales, Profit, Discount, and Quantity.

## Phase 2 - OOP Structure
Define small classes for loading, cleaning, feature engineering, analysis, visualization, and reporting. Later phases will fill in the detailed methods.

In [ ]:
class DataLoader:
    def __init__(self, path):
        self.path = Path(path)

    def load(self):
        try:
            data = pd.read_csv(self.path, parse_dates=['Order Date', 'Ship Date'])
            logger.info('Loaded %s rows and %s columns from %s', *data.shape, self.path.name)
            return data
        except FileNotFoundError as exc:
            logger.error('Dataset file not found: %s', self.path)
            raise exc
        except Exception as exc:
            logger.error('Could not load dataset: %s', exc)
            raise exc


class DataCleaner:
    def __init__(self, data):
        self.df = data.copy()

    def run_pipeline(self):
        return self.df


class FeatureEngineer:
    def __init__(self, data):
        self.df = data.copy()

    def run_pipeline(self):
        return self.df


class EDAAnalyzer:
    def __init__(self, data):
        self.df = data


class Visualizer:
    def __init__(self, data, charts_dir):
        self.df = data
        self.charts_dir = Path(charts_dir)


class ReportGenerator:
    def __init__(self, data, output_dir):
        self.df = data
        self.output_dir = Path(output_dir)


In [ ]:
loader = DataLoader(RAW_DATA_PATH)
df = loader.load()

cleaner = DataCleaner(df)
feature_engineer = FeatureEngineer(df)
eda = EDAAnalyzer(df)
visualizer = Visualizer(df, CHARTS_DIR)
reporter = ReportGenerator(df, OUTPUT_DIR)

print('Phase 2 OOP objects ready:', type(loader).__name__, type(cleaner).__name__, type(feature_engineer).__name__, type(eda).__name__, type(visualizer).__name__, type(reporter).__name__)

## Phase 3 - Advanced Data Cleaning
Clean missing values, duplicate records, text/date formats, and outliers through reusable `DataCleaner` methods.

In [ ]:
class DataCleaner:
    def __init__(self, data):
        self.df = data.copy()

    def handle_missing_values(self):
        try:
            missing_before = self.df.isna().sum()
            if 'Postal Code' in self.df.columns:
                print("Postal Code dtype after fill:", self.df['Postal Code'].dtype)
            for col in self.df.select_dtypes(include=['number']).columns.drop('Postal Code', errors='ignore'):
                if self.df[col].isna().any():
                    self.df[col] = self.df[col].fillna(self.df[col].median())
            for col in self.df.select_dtypes(include=['object']).columns:
                if self.df[col].isna().any():
                    mode = self.df[col].mode(dropna=True)
                    self.df[col] = self.df[col].fillna(mode.iloc[0] if not mode.empty else 'Unknown')
            logger.info('Missing values before cleaning: %s', int(missing_before.sum()))
            return self
        except Exception as exc:
            logger.error('Missing-value handling failed: %s', exc)
            raise exc

    def remove_duplicates(self):
        try:
            before = len(self.df)
            self.df = self.df.drop_duplicates()
            logger.info('Removed %s duplicate rows', before - len(self.df))
            return self
        except Exception as exc:
            logger.error('Duplicate removal failed: %s', exc)
            raise exc

    def standardize_formats(self):
        try:
            for col in self.df.select_dtypes(include=['object']).columns:
                if col.endswith('ID'):
                    self.df[col] = self.df[col].astype(str).str.strip()
                else:
                    self.df[col] = self.df[col].astype(str).str.strip()
            for col in ['Order Date', 'Ship Date']:
                if col in self.df.columns:
                    try:
                        self.df[col] = pd.to_datetime(self.df[col], errors='raise')
                    except ValueError as exc:
                        logger.error('Date parsing failed for %s: %s', col, exc)
                        continue
            logger.info('Standardized text and date formats')
            return self
        except Exception as exc:
            logger.error('Format standardization failed: %s', exc)
            raise exc

    def flag_outliers(self, columns=None):
        try:
            columns = columns or ['Sales', 'Profit', 'Discount', 'Quantity']
            for col in columns:
                if col not in self.df.columns:
                    continue
                q1, q3 = self.df[col].quantile([0.25, 0.75])
                iqr = q3 - q1
                lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                self.df[f'{col} Outlier'] = ~self.df[col].between(lower, upper)
                logger.info('%s outliers flagged in %s', int(self.df[f'{col} Outlier'].sum()), col)
            return self
        except Exception as exc:
            logger.error('Outlier flagging failed: %s', exc)
            raise exc

    def run_pipeline(self):
        return (self.handle_missing_values()
                    .remove_duplicates()
                    .standardize_formats()
                    .flag_outliers()
                    .df)


In [ ]:
cleaner = DataCleaner(df)
df_clean = cleaner.run_pipeline()

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Remaining missing values:", int(df_clean.isna().sum().sum()))
print("Duplicate rows:", int(df_clean.duplicated().sum()))
df_clean[['Sales Outlier', 'Profit Outlier', 'Discount Outlier', 'Quantity Outlier']].sum()

Outliers were flagged (not removed) to preserve the full dataset for reporting; flags are available for future filtering if needed.

### Cleaning Notes
- Missing `Postal Code` values were left as `NaN` so the column stays numeric.
- Exact duplicate rows were removed if found.
- Text fields were stripped of extra spaces and date fields were parsed as datetime values.
- Outliers were flagged with the IQR method instead of removed, preserving real high-value orders for business analysis.

## Phase 4 - Feature Engineering
Create reusable feature-engineering methods for profit margin, shipping duration, sales performance categories, and simple date features.

In [ ]:
class FeatureEngineer:
    def __init__(self, data):
        self.df = data.copy()

    def add_profit_margin(self):
        try:
            zero_sales = self.df['Sales'].eq(0).sum()
            if zero_sales:
                logger.warning('Found %s zero-Sales rows; Profit Margin set to 0 for those rows', zero_sales)
            self.df['Profit Margin'] = np.where(self.df['Sales'] != 0, self.df['Profit'] / self.df['Sales'], 0)
            logger.info('Added Profit Margin')
            return self
        except Exception as exc:
            logger.error('Profit Margin creation failed: %s', exc)
            raise exc

    def add_shipping_duration(self):
        try:
            self.df['Shipping Duration'] = (self.df['Ship Date'] - self.df['Order Date']).dt.days
            logger.info('Added Shipping Duration')
            return self
        except Exception as exc:
            logger.error('Shipping Duration creation failed: %s', exc)
            raise exc

    def add_sales_performance_category(self):
        try:
            labels = ['Low', 'Medium', 'High']
            self.df['Sales Performance Category'] = pd.qcut(self.df['Sales'], q=3, labels=labels)
            logger.info('Added Sales Performance Category')
            return self
        except ValueError:
            self.df['Sales Performance Category'] = pd.cut(self.df['Sales'], bins=3, labels=['Low', 'Medium', 'High'])
            logger.info('Added Sales Performance Category with equal-width bins')
            return self
        except Exception as exc:
            logger.error('Sales Performance Category creation failed: %s', exc)
            raise exc

    def add_date_features(self):
        try:
            self.df['Order Year'] = self.df['Order Date'].dt.year
            self.df['Order Month'] = self.df['Order Date'].dt.to_period('M').astype(str)
            logger.info('Added Order Year and Order Month')
            return self
        except Exception as exc:
            logger.error('Date feature creation failed: %s', exc)
            raise exc

    def run_pipeline(self):
        return (self.add_profit_margin()
                    .add_shipping_duration()
                    .add_sales_performance_category()
                    .add_date_features()
                    .df)


In [ ]:
feature_engineer = FeatureEngineer(df_clean)
df_enriched = feature_engineer.run_pipeline()

new_columns = ['Profit Margin', 'Shipping Duration', 'Sales Performance Category', 'Order Year', 'Order Month']
print("Enriched shape:", df_enriched.shape)
print(df_enriched[new_columns].head())
print("\nSales Performance Category counts:")
print(df_enriched['Sales Performance Category'].value_counts())

### Feature Notes
- `Profit Margin` measures profit relative to sales and safely handles zero sales.
- `Shipping Duration` measures delivery time in days between order and ship dates.
- `Sales Performance Category` splits orders into Low, Medium, and High sales groups using quantiles.
- `Order Year` and `Order Month` support trend analysis in later phases.

## Phase 5 - Exploratory Data Analysis
Use reusable analysis methods for numeric distributions, categorical performance, time trends, and correlation analysis.

In [ ]:
class EDAAnalyzer:
    def __init__(self, data):
        self.df = data.copy()

    def numeric_summary(self):
        cols = ['Sales', 'Profit', 'Discount', 'Quantity', 'Profit Margin', 'Shipping Duration']
        return self.df[cols].describe().T

    def category_summary(self, group_col):
        try:
            return (self.df.groupby(group_col, observed=True)
                    .agg(Orders=('Order ID', 'nunique'),
                         Sales=('Sales', 'sum'),
                         Profit=('Profit', 'sum'),
                         Average_Discount=('Discount', 'mean'),
                         Average_Shipping_Duration=('Shipping Duration', 'mean'))
                    .sort_values('Sales', ascending=False))
        except Exception as exc:
            logger.error('Category summary failed for %s: %s', group_col, exc)
            raise exc

    def monthly_trend(self):
        try:
            trend = (self.df.groupby('Order Month', observed=True)
                     .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'), Orders=('Order ID', 'nunique'))
                     .reset_index())
            trend['Order Month'] = pd.to_datetime(trend['Order Month'])
            return trend.sort_values('Order Month')
        except Exception as exc:
            logger.error('Monthly trend analysis failed: %s', exc)
            raise exc

    def correlation_matrix(self):
        cols = ['Sales', 'Profit', 'Discount', 'Quantity', 'Profit Margin', 'Shipping Duration']
        return self.df[cols].corr(numeric_only=True)


In [ ]:
eda = EDAAnalyzer(df_enriched)
numeric_summary = eda.numeric_summary()
numeric_summary

In [ ]:
category_performance = eda.category_summary('Category')
region_performance = eda.category_summary('Region')
segment_performance = eda.category_summary('Segment')
category_performance

In [ ]:
subcategory_performance = eda.category_summary('Sub-Category')
subcategory_performance.head(10)

In [ ]:
monthly_trend = eda.monthly_trend()
monthly_trend.head()

In [ ]:
correlation_matrix = eda.correlation_matrix()
correlation_matrix

### EDA Notes
- Numeric summaries show sales and profit are highly skewed, so medians and outlier flags matter.
- Category, region, and segment summaries identify where revenue and profit are concentrated.
- Monthly trend data prepares the time-series visualization and business trend discussion.
- The correlation matrix supports the required statistical analysis before visualization.

## Phase 6 - Visualizations
Create and automatically save at least eight visual reports using Matplotlib and Seaborn.

In [ ]:
class Visualizer:
    def __init__(self, data, charts_dir):
        self.df = data.copy()
        self.charts_dir = Path(charts_dir)
        self.charts_dir.mkdir(parents=True, exist_ok=True)

    def _save(self, filename):
        path = self.charts_dir / filename
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        return path

    def sales_distribution(self):
        plt.figure()
        sns.histplot(self.df['Sales'], bins=40, kde=True, color='steelblue')
        plt.title('Sales Distribution')
        plt.xlabel('Sales')
        return self._save('01_sales_distribution.png')

    def profit_boxplot(self):
        plt.figure()
        sns.boxplot(x=self.df['Profit'], color='indianred')
        plt.title('Profit Distribution and Outliers')
        return self._save('02_profit_boxplot.png')

    def sales_by_category(self):
        data = self.df.groupby('Category', observed=True)['Sales'].sum().sort_values(ascending=False)
        plt.figure()
        sns.barplot(x=data.index, y=data.values, hue=data.index, palette='Set2', legend=False)
        plt.title('Sales by Category')
        plt.ylabel('Total Sales')
        return self._save('03_sales_by_category.png')

    def profit_by_subcategory(self):
        data = self.df.groupby('Sub-Category', observed=True)['Profit'].sum().sort_values()
        plt.figure(figsize=(10, 7))
        sns.barplot(x=data.values, y=data.index, hue=data.index, palette='coolwarm', legend=False)
        plt.title('Profit by Sub-Category')
        plt.xlabel('Total Profit')
        return self._save('04_profit_by_subcategory.png')

    def monthly_sales_trend(self):
        trend = self.df.groupby('Order Month', observed=True)['Sales'].sum().reset_index()
        trend['Order Month'] = pd.to_datetime(trend['Order Month'])
        plt.figure(figsize=(11, 5))
        sns.lineplot(data=trend, x='Order Month', y='Sales', marker='o', color='seagreen')
        plt.title('Monthly Sales Trend')
        plt.xticks(rotation=45)
        return self._save('05_monthly_sales_trend.png')

    def correlation_heatmap(self):
        cols = ['Sales', 'Profit', 'Discount', 'Quantity', 'Profit Margin', 'Shipping Duration']
        plt.figure(figsize=(8, 6))
        sns.heatmap(self.df[cols].corr(), annot=True, cmap='vlag', center=0, fmt='.2f')
        plt.title('Correlation Heatmap')
        return self._save('06_correlation_heatmap.png')

    def sales_profit_scatter(self):
        plt.figure()
        sns.scatterplot(data=self.df, x='Sales', y='Profit', hue='Category', alpha=0.65)
        plt.title('Sales vs Profit by Category')
        return self._save('07_sales_profit_scatter.png')

    def performance_category_counts(self):
        plt.figure()
        sns.countplot(data=self.df, x='Sales Performance Category', hue='Sales Performance Category', palette='Set1', legend=False)
        plt.title('Sales Performance Category Counts')
        return self._save('08_sales_performance_counts.png')

    def shipping_duration_by_mode(self):
        plt.figure()
        sns.boxplot(data=self.df, x='Ship Mode', y='Shipping Duration', hue='Ship Mode', palette='Set3', legend=False)
        plt.title('Shipping Duration by Ship Mode')
        plt.xticks(rotation=20)
        return self._save('09_shipping_duration_by_mode.png')

    def create_all(self):
        try:
            paths = [
                self.sales_distribution(),
                self.profit_boxplot(),
                self.sales_by_category(),
                self.profit_by_subcategory(),
                self.monthly_sales_trend(),
                self.correlation_heatmap(),
                self.sales_profit_scatter(),
                self.performance_category_counts(),
                self.shipping_duration_by_mode(),
            ]
            logger.info('Saved %s charts to %s', len(paths), self.charts_dir)
            return paths
        except Exception as exc:
            logger.error('Visualization generation failed: %s', exc)
            raise exc


In [ ]:
visualizer = Visualizer(df_enriched, CHARTS_DIR)
chart_paths = visualizer.create_all()
print(f"Saved {len(chart_paths)} charts:")
for path in chart_paths:
    print(path)

### Visualization Notes
- Nine charts were created, exceeding the minimum requirement of eight.
- The charts cover distributions, outliers, category performance, sub-category profit, time trends, correlations, sales-profit relationships, performance categories, and shipping duration.

## Phase 7 - Automated KPI Summary & Report
Generate KPI summaries programmatically and export a readable text report.

In [ ]:
class ReportGenerator:
    def __init__(self, data, output_dir):
        self.df = data.copy()
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def kpis(self):
        total_sales = self.df['Sales'].sum()
        total_profit = self.df['Profit'].sum()
        return {
            'Total Sales': total_sales,
            'Total Profit': total_profit,
            'Overall Profit Margin': total_profit / total_sales if total_sales else 0,
            'Average Shipping Duration': self.df['Shipping Duration'].mean(),
            'Total Orders': self.df['Order ID'].nunique(),
            'Total Customers': self.df['Customer ID'].nunique(),
        }

    def top_items(self, group_col, metric, n=5, ascending=False):
        return (self.df.groupby(group_col, observed=True)[metric]
                .sum()
                .sort_values(ascending=ascending)
                .head(n))

    def build_report(self):
        try:
            kpis = self.kpis()
            best_region = self.top_items('Region', 'Profit', n=1).index[0]
            worst_region = self.top_items('Region', 'Profit', n=1, ascending=True).index[0]
            best_segment = self.top_items('Segment', 'Profit', n=1).index[0]

            lines = [
                'Mini-Project 1 - Automated KPI Summary',
                '=' * 43,
                f"Total Sales: ${kpis['Total Sales']:,.2f}",
                f"Total Profit: ${kpis['Total Profit']:,.2f}",
                f"Overall Profit Margin: {kpis['Overall Profit Margin']:.2%}",
                f"Average Shipping Duration: {kpis['Average Shipping Duration']:.2f} days",
                f"Total Orders: {kpis['Total Orders']:,}",
                f"Total Customers: {kpis['Total Customers']:,}",
                '',
                f"Best Region by Profit: {best_region}",
                f"Worst Region by Profit: {worst_region}",
                f"Best Segment by Profit: {best_segment}",
                '',
                'Top 5 Sub-Categories by Sales:',
            ]
            lines += [f'- {name}: ${value:,.2f}' for name, value in self.top_items('Sub-Category', 'Sales').items()]
            lines += ['', 'Top 5 Sub-Categories by Profit:']
            lines += [f'- {name}: ${value:,.2f}' for name, value in self.top_items('Sub-Category', 'Profit').items()]
            lines += ['', 'Bottom 5 Sub-Categories by Profit:']
            lines += [f'- {name}: ${value:,.2f}' for name, value in self.top_items('Sub-Category', 'Profit', ascending=True).items()]
            return '\n'.join(lines)
        except Exception as exc:
            logger.error('Report generation failed: %s', exc)
            raise exc

    def save_report(self, filename='report_summary.txt'):
        report = self.build_report()
        path = self.output_dir / filename
        path.write_text(report, encoding='utf-8')
        logger.info('Saved report to %s', path)
        return path, report


In [ ]:
reporter = ReportGenerator(df_enriched, OUTPUT_DIR)
report_path, report_text = reporter.save_report()
print(report_text)

### Report Notes
- The KPI report is generated automatically from the enriched dataset.
- The saved report includes total sales, total profit, overall profit margin, shipping duration, order/customer counts, and top/bottom sub-category performance.

## Phase 8 - Export & Performance Optimization
Export the cleaned/enriched dataset, optimize memory usage, and verify output files.

In [ ]:
def optimize_memory(data):
    optimized = data.copy()
    before = optimized.memory_usage(deep=True).sum()

    for col in optimized.select_dtypes(include=['int']).columns:
        optimized[col] = pd.to_numeric(optimized[col], downcast='integer')

    for col in optimized.select_dtypes(include=['float']).columns:
        optimized[col] = pd.to_numeric(optimized[col], downcast='float')

    for col in optimized.select_dtypes(include=['object']).columns:
        unique_ratio = optimized[col].nunique(dropna=False) / len(optimized)
        if unique_ratio < 0.5:
            optimized[col] = optimized[col].astype('category')

    after = optimized.memory_usage(deep=True).sum()
    reduction = (before - after) / before if before else 0
    return optimized, before, after, reduction


In [ ]:
try:
    cleaned_data_path = OUTPUT_DIR / 'cleaned_data.csv'
    df_enriched.to_csv(cleaned_data_path, index=False)
    logger.info('Exported cleaned dataset to %s', cleaned_data_path)

    df_optimized, memory_before, memory_after, memory_reduction = optimize_memory(df_enriched)
    optimized_data_path = OUTPUT_DIR / 'cleaned_data_optimized.csv'
    df_optimized.to_csv(optimized_data_path, index=False)
except Exception as exc:
    logger.error('Export or optimization failed: %s', exc)
    raise exc

print(f"Memory before optimization: {memory_before / 1024**2:.2f} MB")
print(f"Memory after optimization: {memory_after / 1024**2:.2f} MB")
print(f"Memory reduction: {memory_reduction:.2%}")
print("Cleaned data:", cleaned_data_path)
print("Optimized data:", optimized_data_path)

In [ ]:
expected_outputs = [
    cleaned_data_path,
    optimized_data_path,
    OUTPUT_DIR / 'report_summary.txt',
    *chart_paths,
]

output_status = pd.DataFrame({
    'Output': [path.name for path in expected_outputs],
    'Exists': [path.exists() for path in expected_outputs],
    'Size KB': [round(path.stat().st_size / 1024, 2) if path.exists() else 0 for path in expected_outputs],
})
output_status

### Export and Optimization Notes
- The enriched cleaned dataset is exported as `output/cleaned_data.csv`.
- An optimized copy is exported as `output/cleaned_data_optimized.csv` after downcasting numeric columns and converting repetitive text columns to categories.
- The output check confirms the CSV files, KPI report, and chart images exist.

## Phase 9 - Documentation and Conclusion
This notebook followed the full workflow required for the mini-project: setup, data import, inspection, OOP design, cleaning, feature engineering, EDA, visualization, KPI reporting, export, and optimization.

### Workflow Summary
1. Loaded the Superstore Excel dataset with exception handling.
2. Inspected shape, metadata, missing values, duplicates, data types, and descriptive statistics.
3. Built reusable OOP classes for loading, cleaning, feature engineering, EDA, visualization, and reporting.
4. Cleaned missing postal codes, standardized text/date formats, removed duplicate records if present, and flagged IQR outliers.
5. Added `Profit Margin`, `Shipping Duration`, `Sales Performance Category`, `Order Year`, and `Order Month`.
6. Performed EDA using numeric summaries, grouped performance tables, monthly trends, and correlation analysis.
7. Created and saved nine visualizations.
8. Generated an automated KPI report and exported cleaned/optimized datasets.

In [ ]:
final_summary = {
    'Rows': len(df_enriched),
    'Columns': df_enriched.shape[1],
    'Total Sales': df_enriched['Sales'].sum(),
    'Total Profit': df_enriched['Profit'].sum(),
    'Profit Margin': df_enriched['Profit'].sum() / df_enriched['Sales'].sum(),
    'Average Shipping Duration': df_enriched['Shipping Duration'].mean(),
    'Charts Created': len(chart_paths),
}
pd.Series(final_summary)

### Final Business Insights
- Technology generated the highest sales and strongest category profit, making it the main revenue and profit driver.
- Furniture produced high sales but much lower profit than Technology and Office Supplies, so pricing, discounting, or cost structure should be reviewed.
- The West region was the strongest profit region, while the Central region had the weakest profit performance.
- Tables, Bookcases, and Supplies were the weakest sub-categories by profit and need closer margin control.
- Discount has a strong negative relationship with Profit Margin, so discount policy is a major profitability risk.
- Average shipping duration was about four days, with all shipments between zero and seven days.

### Project Completion Checklist
- Import and inspect dataset: completed.
- Advanced cleaning: completed.
- Reusable preprocessing functions/classes: completed.
- Feature engineering: completed.
- Modular OOP workflow: completed.
- Advanced EDA and statistical analysis: completed.
- Minimum eight visualizations: completed with nine charts.
- Automated KPI summary/report: completed.
- Exported cleaned dataset and visual reports: completed.
- Memory/performance optimization: completed.
- Full notebook documentation: completed.